# Домашнее задание 2: DWH-ассистент на LangChain с памятью фактов

Агент сохраняет **факты**, а не историю сообщений: имя, стек, уровень, цели, отрасль, источники и частоту обновления. Файл `memory.json` читается при старте; каждый `user_id` имеет отдельный профиль.

Что сдаётся: этот Notebook с кодом и выводом; [пример файла памяти](examples/memory.json); [демонстрация двух отдельных процессов](evidence/memory_restart.json).

Модель: **Qwen3-4B-Instruct-2507 Q4_K_M**. Весь инференс локальный; также поддержаны профили 1.5B/3B и OpenRouter из проекта. Сведения в демонстрации вымышленные.

## Подготовка

Из корня проекта выполните команды раздела «Задание 2» в [README](README.md): создайте `.venv`, установите `requirements.txt`, выполните `setup.ps1`, зарегистрируйте ядро `dwh-assistant` и откройте Notebook этим ядром.

Для интерактивного чата выполните ячейки 1–4 по порядку. Служебная CELL и основной чат — разные блоки вывода. Ни JSON извлечения, ни диагностические события не выводятся в основной чат.

Подсказки задания про `ConversationChain`/`BaseChatMemory` относятся к старому API. Здесь используется современная LCEL-цепочка `prompt | RunnableLambda(LLM) | parser` и собственный адаптер `FileFactMemory` с методами `load_memory_variables()` и `save_context()`. `RunnableWithMessageHistory` не нужен: история сообщений в долгосрочную память не записывается. [Справочник LangChain RunnableSequence](https://reference.langchain.com/python/langchain-core/runnables/base/RunnableSequence).

## 1. Импорты и путь проекта

In [1]:
import atexit
import json
import os
from pathlib import Path
import sys
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dwh_agent").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Откройте Notebook из папки проекта dwh-assistant")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from dwh_agent.fact_agent import FactAgent
from dwh_agent.fact_memory import FileFactMemory
from dwh_agent.local_server import LocalServer
from dwh_agent.notebook_ui import NotebookChat
from verify_memory_restart import run_restart_demo

## 2. Собственная служебная CELL

В этой ячейке появляются загрузка/сохранение памяти, состояние модели и ошибки. Чат находится ниже.

In [2]:
ui = NotebookChat()
display(ui.service_output)

Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…

## 3. Конфигурация и запуск

Для другого человека измените `USER_ID`. Если модели уже скачаны в другой папке, задайте `ASSETS_DIR` или переменную окружения `DWH_ASSETS_DIR`. Рабочий `memory.json` исключён из Git. Для OpenRouter задайте ключ в окружении терминала до запуска Jupyter.

In [3]:
BACKEND = "local-4b"
USER_ID = "student"
MEMORY_PATH = ROOT / "memory.json"
ASSETS_DIR = Path(os.environ.get("DWH_ASSETS_DIR", str(ROOT)))
CPU = False
START_LOCAL_SERVER = BACKEND.startswith("local-")

if "server" in globals() and server is not None:
    server.close()
server = None
try:
    if START_LOCAL_SERVER:
        ui.service({"event": "server_starting", "backend": BACKEND})
        server = LocalServer(BACKEND, assets_dir=ASSETS_DIR, cpu=CPU).start()
        atexit.register(server.close)
    memory = FileFactMemory(MEMORY_PATH, USER_ID)
    agent = FactAgent(memory, backend=BACKEND, on_event=ui.service)
    ui.attach(agent, on_close=server.close if server else None)
    ui.service({"event": "ready", "backend": BACKEND, "user_id": USER_ID})
except Exception as exc:
    if server is not None:
        server.close()
    ui.service({"event": "startup_error", "message": str(exc)})
    raise

## 4. Основной чат

Введите сведения о себе и задайте вопрос по DWH. При просьбе продолжить знакомство агент задаёт один следующий вопрос и учитывает уже заполненные поля.

Команды: `/memory` — показать факты в служебной CELL; `/forget stack` — удалить одно поле; `/clear` — очистить только текущий профиль; `/exit` — закрыть чат и остановить принадлежащий ему сервер, сохранив память.

Ручная проверка: сообщите имя и стек → `/exit` → перезапустите ядро → выполните ячейки 1–4 с тем же `USER_ID` → спросите «Что ты обо мне помнишь?». Для замены поля сообщайте полное новое значение, например «Мой стек теперь — Python и ClickHouse».

In [4]:
display(ui.chat_box)
ui.chat_output.append_stdout("Ассистент: " + (memory.next_question() or "Профиль загружен. Какой вопрос по DWH обсудим?") + "\n")

## 5. Демонстрация настоящего перезапуска

Следующая ячейка запускает **два отдельных Python-процесса**. Первый извлекает факты и завершается. Второй создаёт новый агент и читает факты из файла. Между процессами не передаётся история сообщений или объект памяти.

Демонстрация сбрасывает только профиль `restart_demo` в `examples/memory.json`; рабочий `memory.json` не затрагивается. Сервер модели должен работать (при необходимости повторите ячейку 3). Фактические ответы печатаются без ручной редакции.

In [5]:
demo = run_restart_demo(ROOT / "examples/memory.json", backend=BACKEND, start_server=False)
(ROOT / "evidence/memory_restart.json").write_text(json.dumps(demo, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
first_run, second_run = demo["sessions"]
print("ПЕРВЫЙ ЗАПУСК")
print("Пользователь:", first_run["message"])
print("Ассистент:", first_run["answer"])
print("\nПОСЛЕ ЗАВЕРШЕНИЯ ПЕРВОГО ПРОЦЕССА И НОВОГО ЗАПУСКА")
print("Пользователь:", second_run["message"])
print("Ассистент:", second_run["answer"])

ПЕРВЫЙ ЗАПУСК
Пользователь: Меня зовут Алексей. Мой стек — Python и PostgreSQL. Мой уровень — начинающий аналитик. Моя цель — спроектировать DWH для автосалона. Предметная область — автосалон.
Ассистент: Какие источники данных доступны?

ПОСЛЕ ЗАВЕРШЕНИЯ ПЕРВОГО ПРОЦЕССА И НОВОГО ЗАПУСКА
Пользователь: Как меня зовут, какой у меня стек и какая цель? Затем задай следующий вопрос анкеты.
Ассистент: Вас зовут Алексей, ваш стек — Python и PostgreSQL, ваша цель — спроектировать DWH для автосалона.  

Следующий вопрос анкеты: Какие источники данных доступны?


## 6. Отдельная служебная CELL демонстрации

Здесь доступны проверки, PID процессов и фактический файл памяти. Этот блок нужен также для просмотра результатов в GitHub, где интерактивные widgets могут не отображаться.

In [6]:
print(json.dumps({
    "passed": demo["passed"],
    "checks": demo["checks"],
    "process_ids": [row["pid"] for row in demo["sessions"]],
    "memory": demo["memory"],
}, ensure_ascii=False, indent=2))
assert demo["passed"], "Смотрите реальные результаты в evidence/memory_restart.json"

{
  "passed": true,
  "checks": {
    "different_processes": true,
    "first_started_empty": true,
    "facts_extracted": true,
    "loaded_from_file": true,
    "question_did_not_change_facts": true,
    "recall_mentions_facts": true,
    "extraction_succeeded": true
  },
  "process_ids": [
    26896,
    6320
  ],
  "memory": {
    "version": 1,
    "profiles": {
      "restart_demo": {
        "facts": {
          "name": "Алексей",
          "stack": "Python и PostgreSQL",
          "level": "начинающий аналитик",
          "goals": "спроектировать DWH для автосалона",
          "domain": "автосалон"
        }
      }
    }
  }
}


## 7. Как устроен код

- `dwh_agent/fact_agent.py`: две LangChain-цепочки — извлечение JSON и ответ по фактам.
- `dwh_agent/fact_memory.py`: схема фактов, проверка цитат, атомарная запись, блокировка файла и отдельные профили.
- `dwh_agent/notebook_ui.py`: чат и собственная служебная CELL.
- `dwh_agent/transport.py`, `local_server.py`, `protocol.py`, `event_log.py`: общие компоненты после рефакторинга; первый REPL продолжает работать.
- `test_fact_memory.py`: проверки перезапуска объекта, профилей, исправления/удаления фактов, ошибок JSON/диска, LangChain и разделения вывода.

Извлекатель видит только последнее сообщение пользователя. Сохранённые факты передаются только в цепочку ответа, чтобы модель не выдавала старое за вновь сообщённое. Значение и подтверждающая цитата должны присутствовать в текущем сообщении. Некорректное извлечение повторяется один раз; затем память сохраняется без изменений, а событие уходит в служебную CELL.

Проверка цитат не доказывает смысловую правильность. Модель 4B может ошибиться в отнесении фразы к полю или в ответе по DWH. Профиль можно проверить и исправить. В память не пишутся сообщения и ответы; во время живого чата текст отображается в widget. Демонстрационные ответы отдельно сохранены как доказательство выполнения задания.

## Завершение

Для ручного завершения отправьте `/exit` в чат. При проверке Notebook через nbclient сервер автоматически закрывается после выполнения всех ячеек.

In [7]:
if os.environ.get("DWH_NOTEBOOK_VERIFY") == "1":
    if server is not None:
        server.close()
    ui.service({"event": "verification_complete", "server_stopped": True})